In [ ]:
import pandas as pd
from transformers import pipeline

In [ ]:
# ==============================================================================
# 1. Datasets synthétiques générés inline (3 types de données cyber)
# ==============================================================================

# Dataset 1 : Phishing (Emails / Messages)
df_phishing = pd.DataFrame({
    "text": [
        "URGENT: Your payroll direct deposit failed. Click here to re-enter your banking credentials immediately.",
        "Team, please find attached the meeting notes and project timeline for next sprint.",
        "Your Microsoft 365 password expires in 2 hours. Keep current password by validating at login-update-auth.com",
        "Hi, can you review the attached quarterly budget spreadsheet before the 3 PM call?"
    ],
    "ground_truth": ["Phishing", "Legitimate", "Phishing", "Legitimate"]
})

# Dataset 2 : URLs (Syntaxe web, chemins, domaines suspects)
df_urls = pd.DataFrame({
    "text": [
        "https://accounts.google.com/signin/v2/identifier",
        "http://paypal-security-update.account-verification.cc/login.php?token=928374",
        "https://github.com/torvalds/linux/commit/a1b2c3d4e5f6",
        "http://192.168.1.105:8080/exploit.bin"
    ],
    "ground_truth": ["Legitimate", "Malicious URL", "Legitimate", "Malicious URL"]
})

# Dataset 3 : Logs système (Syslog / Auth / Web logs)
df_logs = pd.DataFrame({
    "text": [
        "sshd[12435]: Accepted publickey for ubuntu from 10.0.0.15 port 54322 ssh2",
        "sshd[12480]: Failed password for root from 185.220.101.5 port 39124 ssh2 - 45 failures in 10s",
        "kernel: [10293.45] eth0: link up, 1000Mbps, full-duplex",
        "nginx: 172.16.0.4 - - [POST /admin/upload.php?cmd=whoami HTTP/1.1] 200 4521"
    ],
    "ground_truth": ["Normal Log", "Suspicious Log", "Normal Log", "Suspicious Log"]
})

In [ ]:
# ==============================================================================
# 2. Pipeline unifié : Zero-Shot Classification
# ==============================================================================
# Le modèle évalue l'adéquation sémantique sans fine-tuning préalable
classifier = pipeline(
    "zero-shot-classification", 
    model="valhalla/distilbart-mnli-12-3"
)

# Configuration des tâches : labels candidats adaptés à chaque modalité
tasks = [
    {
        "name": "PHISHING DETECTION (Texte naturel)",
        "df": df_phishing,
        "candidate_labels": ["phishing email scam", "legitimate business communication"],
        "label_mapping": {
            "phishing email scam": "Phishing",
            "legitimate business communication": "Legitimate"
        }
    },
    {
        "name": "MALICIOUS URL DETECTION (Syntaxe Web)",
        "df": df_urls,
        "candidate_labels": ["malicious phishing url attack", "benign trusted website url"],
        "label_mapping": {
            "malicious phishing url attack": "Malicious URL",
            "benign trusted website url": "Legitimate"
        }
    },
    {
        "name": "LOG ANOMALY DETECTION (Données semi-structurées)",
        "df": df_logs,
        "candidate_labels": ["security breach attack event", "normal system activity log"],
        "label_mapping": {
            "security breach attack event": "Suspicious Log",
            "normal system activity log": "Normal Log"
        }
    }
]

In [ ]:
# ==============================================================================
# 3. Exécution et comparaison des résultats
# ==============================================================================
for task in tasks:
    print("=" * 80)
    print(f"TÂCHE : {task['name']}")
    print("=" * 80)
    
    texts = task["df"]["text"].tolist()
    labels = task["candidate_labels"]
    mapping = task["label_mapping"]
    
    # Inférence sur le batch
    results = classifier(texts, candidate_labels=labels)
    
    # Formatage des sorties
    for i, res in enumerate(results):
        raw_text = res["sequence"]
        top_label = res["labels"][0]
        top_score = res["scores"][0]
        
        predicted_class = mapping[top_label]
        expected_class = task["df"].iloc[i]["ground_truth"]
        
        match_status = "[OK]" if predicted_class == expected_class else "[ERREUR]"
        
        # Tronquer le texte si trop long pour l'affichage
        disp_text = raw_text if len(raw_text) < 70 else raw_text[:67] + "..."
        
        print(f"{match_status} Attendu: {expected_class:<15} | Prédit: {predicted_class:<15} (confiance: {top_score:.2f})")
        print(f"     Entrée : {disp_text}\n")